In [18]:
import os
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey, Table
from sqlalchemy.orm import declarative_base, relationship, sessionmaker

# 1. DELETE THE OLD DB FILE (Fresh start every time)
db_name = "travel_planner.db"
if os.path.exists(db_name):
    os.remove(db_name)
    print(f"Deleted old {db_name} file.")

# 2. SETUP THE ENGINE
engine = create_engine(f"sqlite:///{db_name}", echo=False)
Base = declarative_base()

Deleted old travel_planner.db file.


In [ ]:
# Bridge linking City <-> Activity
city_activity_bridge = Table(
    'city_activity_bridge', Base.metadata,
    Column('city_id', Integer, ForeignKey('cities.id')),
    Column('activity_id', Integer, ForeignKey('activities.id'))
)

# Bridge linking Country <-> Activity
country_activity_bridge = Table(
    'country_activity_bridge', Base.metadata,
    Column('country_id', Integer, ForeignKey('countries.id')),
    Column('activity_id', Integer, ForeignKey('activities.id'))
)

class Country(Base):
    __tablename__ = 'countries'
    
    id = Column(Integer, primary_key=True)
    name = Column(String)

    # A Country has many Cities (One-to-Many)
    cities = relationship('City', back_populates='country')
    
    # A Country has many Activities (Many-to-Many using the bridge)
    activities = relationship('Activity', secondary=country_activity_bridge, back_populates='countries')


class City(Base):
    __tablename__ = 'cities'
    
    id = Column(Integer, primary_key=True)
    name = Column(String)
    
    # The Foreign Key linking to the Country table
    country_id = Column(Integer, ForeignKey('countries.id'))
    country = relationship('Country', back_populates='cities')

    # A City has many Activities (Many-to-Many using the bridge)
    activities = relationship('Activity', secondary=city_activity_bridge, back_populates='cities')


class Activity(Base):
    __tablename__ = 'activities'
    
    id = Column(Integer, primary_key=True)
    name = Column(String)

    # Links back to both Country and City so you can search from either direction!
    countries = relationship('Country', secondary=country_activity_bridge, back_populates='activities')
    cities = relationship('City', secondary=city_activity_bridge, back_populates='activities')


# Create the blank tables
Base.metadata.create_all(engine)
print("New database and tables created!")

# Start a session to add data
Session = sessionmaker(bind=engine)
session = Session()

# Create standard data
italy = Country(name="Italy")
japan = Country(name="Japan")

rome = City(name="Rome", country=italy)
tokyo = City(name="Tokyo", country=japan)

cooking_class = Activity(name="Cooking Class")
museum_tour = Activity(name="Museum Tour")

rome.activities.append(cooking_class)
rome.activities.append(museum_tour)

# Connecting an activity directly to a Country
italy.activities.append(cooking_class) 
japan.activities.append(museum_tour)

# Save it all
session.add_all([italy, japan, rome, tokyo, cooking_class, museum_tour])
session.commit()

print("Database populated successfully!")

In [25]:
Session = sessionmaker(bind=engine)
session = Session() 

# Create standard data
# italy = Country(name="Italy")
spain = Country(name="Spain")
france = Country(name="France")
germany = Country(name="Germany")

barcelona = City(name="Barcelona")
frankfurt = City(name="Frankfurt")
valencia = City(name="Valenica")

sightseeing = Activity(name="Sightseeing")
session.add_all([spain, france, germany, barcelona, frankfurt, valencia])
session.commit()

print("Database pupulated successfully!")


Database pupulated successfully!
